# Data Preparation

This notebook prepares the data for the rest of the thesis, following the decisions of the meeting with the supervisor (07/10/2026). The analysis that led to these decisions is in `Data_Processing.ipynb`.

The shared code (paths, helpers, signal readers and the recording index) lives in `src/multiscope.py`, so from this notebook on, they will use it instead of copying each cell repeatedly. 

## Table of contents

1. [Setup](#1-setup)
2. [Decisions of the meeting](#2-decisions-of-the-meeting)
3. [Recording index](#3-recording-index)
4. [Zero-filled packets](#4-zero-filled-packets)
5. [Signal quality per recording](#5-signal-quality-per-recording)
    - 5.1. [Repeated recordings: is the last one better?](#51-repeated-recordings-is-the-last-one-better)
6. [Clinical database (18.09.2026)](#6-clinical-database-18092026)
    - 6.1. [Comparison with the previous version (21.07.2026)](#61-comparison-with-the-previous-version-21072026)
    - 6.2. [Patients with signals and clinical data](#62-patients-with-signals-and-clinical-data)
    - 6.3. [Missing data](#63-missing-data)
    - 6.4. [Patient names in the reports](#64-patient-names-in-the-reports)
7. [12-lead ECGs](#7-12-lead-ecgs)

...



### 1. Setup
[[go back to the top]](#table-of-contents)

`autoreload` reloads `multiscope.py` automatically whenever it is edited, so the kernel does not need to be restarted.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the project root importable, wherever the notebook is
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.multiscope import *

print("Dataset root:", DATA_ROOT)
print("Clinical DB:", next(DB_DIR.glob(CLINICAL_DB_GLOB)).name)

Dataset root: C:\Users\aleja\OneDrive\Documentos\GitHub\cardio-trimodal-align\data\DatasetCHVNGE
Clinical DB: 18.09.2026_DB MultiScope Class HTP Atualizado (2).xlsx


### 2. Decisions of the meeting
[[go back to the top]](#table-of-contents)

- **Clinical database:** the updated version from `18.09.2026` is used. It may include patients whose signals were not shared (recorded with PCG, ECG and PPG).

- **Patients without usable signals:** 71 to 77, 131, 132, 273, 274 and 275.

- **Patients recorded with both apps** (118, 119, 121, 122): only their new app recordings are used.

- **Not used:** `Samsung_12_lead_ECG`, `compiled_dataset.pkl`, `Old_12_lead_ECG`, `ECGs` (the `.pkl` files in `New_12_lead_ECG` are a compiled version of them) and `Patients_Multiscope`. They were moved to `Unused/` by `scripts/organize_data.py`.

- **Whole signals:** the signals are used whole. Lost packets can be ignored, according to the supervisor. The data is sent via Bluetooth, and the timestamps record when each packet was received.

- **Low quality scores are expected:** the device is very sensitive, so any small movement of the doctor or the patient affects the signal. We can see that example in patient 168, in particular `P168_Tricúspide_PCG_2024-06-12;12-19-49.9170` and the `P168_Tricúspide_ECG_2024-06-12;12-19-49.9170` where row 7 has only 0's for the PCG and some 0's and real values in ECG, as approached in [[Section 4.]](#4-zero-filled-packets)

- **Repeated recordings:** the last one is usually the best, but this is not a fixed rule. In the quality annotations, `AV1` is the second recording.

### 3. Recording index
[[go back to the top]](#table-of-contents)

`build_index()` builds the same index as Section 7 of `Data_Processing`, but without the patients that have no usable signals and without the Rijuven recordings of patients above 108. The `name` column is the file name of the ECG, within the same patient and site, sorting by it gives the recording order (the new app names contain the date and time, and Rijuven repeats end in `2`, `3`, ...).

In [2]:
recordings = build_index()

print(f"Recordings: {len(recordings)} | patients: {recordings['patient'].nunique()}")
pd.crosstab(recordings["site"], recordings["app"], margins=True)

Recordings: 1350 | patients: 289


app,new_app,rijuven,All
site,,,
AV,245,102,347
MV,229,102,331
PV,232,107,339
TV,227,106,333
All,933,417,1350


### 4. Zero-filled packets
[[go back to the top]](#table-of-contents)

Some new app packets were filled with zeros by the app (314 per signal, see `Data_Processing` Section 8.2.). A flat line at 0 is not a real ECG or PCG value, and it would look like a huge jump to any model, so `load_recording` removes these packets by default, at the same positions in the ECG and the PCG. 

This is consistent with the other lost packets, which also disappear from the signal. The check below loads one recording with zero packets both ways.

In [3]:
row = recordings[recordings["app"] == "new_app"].iloc[0]
for _, row in recordings[recordings["app"] == "new_app"].iterrows():
    ecg, pcg = load_recording(row)
    if ecg.meta["zero_packets"]:
        break

raw_ecg, raw_pcg = load_recording(row, drop_zeros=False)
print(f"Patient {row['patient']}, {row['site']}: {ecg.meta['zero_packets']} zero packets removed")
print(f"ECG: {raw_ecg.duration:.2f} s -> {ecg.duration:.2f} s | PCG: {raw_pcg.duration:.2f} s -> {pcg.duration:.2f} s")

Patient 110, TV: 1 zero packets removed
ECG: 24.25 s -> 24.00 s | PCG: 24.25 s -> 24.00 s


### 5. Signal quality per recording
[[go back to the top]](#table-of-contents)

The quality scores are joined to the recordings. In the annotation file, the first recording of a site is called `AV`, the second `AV1`, the third `AV2`, and so on, so each recording gets the same name from its position within its patient and site.

The quality scores are joined to the recordings by patient and spot. The annotation file uses two naming conventions for repeated recordings, in early patients the first recording is `PV` and the second `PV2`, and in later patients they are `PV1`, `PV2`, `PV3`. In both, `1` (or no number) is the first recording, so `PV1` is renamed `PV`, and each recording is named by its position within its patient and site: `PV`, `PV2`, `PV3`, ...

In [16]:
annotation_file = read_table(QUALITY_DIR / "ULSGE_manual_SQA_ver_2.xlsx")
rank = recordings.groupby(["patient", "site"]).cumcount()                     # 0 = first recording of that site
supervisor_rule = recordings["site"] + rank.map(lambda k: str(k) if k else "")  # AV, AV1, AV2, ...

compare = pd.DataFrame({
    "annotation_file": annotation_file.groupby("Trial")["Spot"].apply(lambda s: sorted(s.str.strip())),
    "supervisor_rule": supervisor_rule.groupby(recordings["patient"]).apply(sorted),
})
compare.loc[[10, 35, 163, 176]]

,annotation_file,supervisor_rule
10,"[AV, MV, PV, PV2, TV, TV2]","[AV, MV, PV, PV1, TV, TV1]"
35,"[AV, MV, MV2, PV, PV2, TV]","[AV, MV, MV1, PV, PV1, TV]"
163,"[AV1, AV2, AV3, MV1, MV2, PV1, PV2, TV]","[AV, AV1, AV2, MV, MV1, PV, PV1, TV]"
176,"[AV1, AV2, AV3, MV1, MV2, PV1, PV2, TV1, TV2]","[AV, AV1, AV2, MV, MV1, PV, PV1, TV, TV1]"


`annotation_file` shows the names used in the annotation file, and `supervisor_rule` the names the recordings would have following the rule mentioned in the meeting (first recording `AV`, second `AV1`, third `AV2`). 

If the rule were right, both columns would match. They do not, in early patients the second recording is called `PV2` (there is no `PV1`), and in later patients the recordings are numbered `AV1`, `AV2`, `AV3` (there is no `AV` without a number). In both conventions the number seems to follow the order of the recordings, with `1` (or no number) being the first. This interpretation is used below, and it has been sent to the supervisor for confirmation.

In [4]:
sqa = read_table(QUALITY_DIR / "ULSGE_manual_SQA_ver_2.xlsx")
sqa = sqa[~sqa["Trial"].isin(NO_SIGNAL_IDS)]
sqa = pd.DataFrame({"patient": sqa["Trial"], "spot": sqa["Spot"].str.strip(),
                    "sqa_ecg": sqa["ECG"], "sqa_pcg": sqa["PCG"], "sqa": sqa["mSQA_min"]})
print("Repeated spots in the annotation file:", sqa.duplicated(["patient", "spot"]).sum())

rank = recordings.groupby(["patient", "site"]).cumcount()                  # 0 = first recording of that site
recordings["spot"] = recordings["site"] + rank.map(lambda k: str(k) if k else "")
recordings = recordings.drop(columns=["sqa_ecg", "sqa_pcg", "sqa"], errors="ignore").merge(sqa, on=["patient", "spot"], how="left")

unmatched = sqa.merge(recordings[["patient", "spot"]], how="left", indicator=True).query("_merge == 'left_only'")
print(f"Recordings with a score: {recordings['sqa'].notna().sum()} of {len(recordings)}")
print("Scores without a recording:", unmatched[["patient", "spot"]].values.tolist())

Repeated spots in the annotation file: 0
Recordings with a score: 1039 of 1350
Scores without a recording: [[10, 'PV2'], [10, 'TV2'], [16, 'PV2'], [16, 'TV2'], [18, 'PV2'], [18, 'TV2'], [35, 'PV2'], [35, 'MV2'], [39, 'PV2'], [39, 'TV2'], [86, 'AV2'], [86, 'PV2'], [101, 'TV2'], [125, 'TV2'], [130, 'PV2'], [135, 'MV2'], [146, 'AV2'], [150, 'AV2'], [150, 'MV2'], [151, 'MV2'], [152, 'PV2'], [155, 'MV2'], [159, 'AV2'], [160, 'AV2'], [161, 'PV2'], [163, 'AV3'], [163, 'MV2'], [163, 'PV2'], [164, 'AV2'], [164, 'PV2'], [165, 'AV2'], [166, 'AV2'], [170, 'MV2'], [170, 'TV2'], [171, 'AV2'], [172, 'AV3'], [172, 'PV2'], [173, 'AV3'], [174, 'AV2'], [176, 'AV3'], [176, 'MV2'], [176, 'PV2'], [176, 'TV2'], [177, 'AV2'], [177, 'MV2'], [177, 'PV2'], [177, 'TV2'], [178, 'AV2'], [179, 'AV2'], [179, 'PV3'], [179, 'TV2'], [180, 'AV2'], [180, 'PV2'], [181, 'TV2'], [182, 'AV2'], [182, 'TV2'], [184, 'AV2'], [185, 'PV2'], [185, 'TV2'], [186, 'PV2'], [187, 'AV2'], [187, 'MV2'], [187, 'PV2'], [187, 'TV2'], [188, 'A

In [9]:
compare = pd.DataFrame({
    "annotations": sqa.groupby("patient")["spot"].apply(sorted),
    "recordings": recordings.groupby("patient")["spot"].apply(sorted),
})
compare.loc[[10, 35, 163, 176]]

,annotations,recordings
patient,,
10,"[AV, MV, PV, PV2, TV, TV2]","[AV, MV, PV, PV1, TV, TV1]"
35,"[AV, MV, MV2, PV, PV2, TV]","[AV, MV, MV1, PV, PV1, TV]"
163,"[AV1, AV2, AV3, MV1, MV2, PV1, PV2, TV]","[AV, AV1, AV2, MV, MV1, PV, PV1, TV]"
176,"[AV1, AV2, AV3, MV1, MV2, PV1, PV2, TV1, TV2]","[AV, AV1, AV2, MV, MV1, PV, PV1, TV, TV1]"


In both, the numbers go up to the number of recordings of that site, so `PV1` (or `PV`) is **assumed** to be the first recording, `PV2` the second, and so on. This assumes the annotator numbered the recordings in the order they were made, it has to be confirmed with the supervisor, who mentioned that `AV1` is the second recording.

#### 5.1. Repeated recordings: is the last one better?
[[go back to the topic]](#5-signal-quality-per-recording)

Before choosing which repeat to keep, this compares the first and the last recording of every repeated site that has scores.

In [5]:
repeated = recordings.dropna(subset=["sqa"]).groupby(["patient", "site"]).filter(lambda g: len(g) > 1)
first_last = repeated.groupby(["patient", "site"])["sqa"].agg(["first", "last"])

np.sign(first_last["last"] - first_last["first"]).map({1: "last is better", 0: "equal", -1: "last is worse"}).value_counts()

last is better    4
equal             4
last is worse     3
Name: count, dtype: int64

### 6. Clinical database (18.09.2026)
[[go back to the top]](#table-of-contents)

#### 6.1. Comparison with the previous version (21.07.2026)
[[go back to the topic]](#6-clinical-database-18092026)

In [5]:
old_db = load_db_version(next(DB_DIR.glob("21.07.2026_DB MultiScope*.xlsx")))
db = load_db_version(next(DB_DIR.glob(CLINICAL_DB_GLOB)))

print(f"Patients: {len(old_db)} -> {len(db)} | columns: {old_db.shape[1]} -> {db.shape[1]}")
print("New patients:", sorted(set(db["ID"]) - set(old_db["ID"])))
print("Removed patients:", sorted(set(old_db["ID"]) - set(db["ID"])))
print("New columns:", list(db.columns.difference(old_db.columns)))
print("Removed columns:", list(old_db.columns.difference(db.columns)))
changed_columns(old_db, db)

Patients: 301 -> 301 | columns: 39 -> 39
New patients: []
Removed patients: []
New columns: []
Removed columns: []


Ritmo                  4
Report_ecoTT           4
FEVE4C                 3
PSAP                   3
FEVEBP                 3
IT                     2
EcoTT                  2
Alt Valv Tricúspide    2
Alt Valv Pulmonar      2
Alt Valv Mitral        2
Alt_Valv_Aortica       2
Fluxo pulmonar         2
protese_aortica        2
IM                     2
IAO                    2
Gradiente VD/AD        2
IP                     2
Peso                   1
Altura                 1
dtype: int64

#### 6.2. Patients with signals and clinical data
[[go back to the topic]](#6-clinical-database-18092026)

In [6]:
signal_ids, db_ids = set(recordings["patient"]), set(db["ID"])

print("In the DB without recordings:", sorted(db_ids - signal_ids))
print("Recordings not in the DB:", sorted(signal_ids - db_ids))
print("Patients with both:", len(signal_ids & db_ids))

In the DB without recordings: [71, 72, 73, 74, 75, 76, 77, 131, 132, 273, 274, 275]
Recordings not in the DB: []
Patients with both: 289


#### 6.3. Missing data
[[go back to the topic]](#6-clinical-database-18092026)

Share of missing values per column, only for the patients with recordings. Empty text and `NA` (used in the database for missing echocardiograms) count as missing.

In [7]:
cohort = db[db["ID"].isin(signal_ids)]
missing = cohort.isna() | cohort.astype(str).apply(lambda s: s.str.strip()).isin(["", "NA", "nan"])

missing.mean().mul(100).round(1).sort_values(ascending=False).to_frame("missing_%")

,missing_%
AoMeanP,99.3
AoSistP,99.3
AoDiastP,99.0
RVP,92.0
mPAP,91.7
pCWP,91.7
Dados Hemodinâmicos,91.0
VMaxRegTric,57.8
HTP Probability,38.8
PSAP,37.7


#### 6.4. Patient names in the reports
[[go back to the topic]](#6-clinical-database-18092026)

Before the reports are sent to any language model, they must not contain the patient's name or any other private information. To find a name inside a report, the code needs to know which name to look for, so the patient list (in `Unused/Patients_Multiscope`) is used for this purpose only. 

Only the `ID` (the study number, 1 to 301, which does not identify anyone outside the study) and `NAME` columns are read, so the health number (NSC) and the other private columns are never loaded. The names stay in memory, are never printed (only counts are), and are deleted at the end of this section.

In [8]:
path = next((UNUSED_DIR / "Patients_Multiscope").glob("*.xlsx"))
names = pd.read_excel(path, header=1, usecols=["ID", "NAME"])                     # only the two columns needed
names["ID"] = pd.to_numeric(names["ID"], errors="coerce")
names = names.dropna(subset=["ID"]).drop_duplicates("ID").set_index("ID")["NAME"]   # never printed

reports = cohort.set_index("ID")[REPORT_COLUMN].dropna()
checked = [pid for pid in reports.index if pid in names.index]
with_name = sum(has_name(reports[pid], names[pid]) for pid in checked)

print(f"Reports: {len(reports)} | checked against the list: {len(checked)} | containing the patient's name: {with_name}")

Reports: 256 | checked against the list: 225 | containing the patient's name: 153


- **Reports (256):** patients with recordings that have something written in `Report_ecoTT`.

- **Checked against the list (225):** reports of patients whose name is in the patient list. The other 31 belong to patients added after the list (Patient_Multiscope 14.01), so their names are unknown.

- **Containing the patient's name (153):** most reports include the patient's name. They are not de-identified, so they must not be printed, pushed to GitHub or sent to an external model until the names are removed.

**Where is the name?** To remove it, its position in the report has to be known. As a first look, the first word of every line of every report is counted, which shows the structure of the reports without showing their content.

In [9]:
first_words = reports.str.split("\n").explode().str.strip().str.split().str[0].str.rstrip(":")
first_words.value_counts().head(20)

Report_ecoTT
Diam              274
Ecocardiograma    244
VE                196
Função            196
Massa             165
F.E.              157
Vol               151
Data              149
Septo             144
Relatório         142
Parede            140
Dilatação         140
Medições          138
Vila              138
Executado         138
Nº                134
Altura            134
Peso              133
Qualidade         133
Índice            133
Name: count, dtype: int64

Each report has a header (`Ecocardiograma`, `Nº`, `Altura`, `Peso`...) followed by the measurements (`Diam`, `VE`, `Massa`, `F.E.`...). `Vila` is the hospital address (Vila Nova de Gaia). There is no line starting with `Nome`, so if the name has a label, it is in the middle of a line.

In [10]:
def name_position(report, name):
    """Line number of the first name match and the word right before it."""
    parts = [p for p in str(name).split() if len(p) >= 4]
    for i, line in enumerate(str(report).split("\n")):
        matches = [m for p in parts for m in re.finditer(rf"\b{re.escape(p)}\b", line, re.IGNORECASE)]
        if matches:
            before = line[:min(m.start() for m in matches)].split()
            return i, before[-1] if before else "(start of line)"
    return None


positions = [name_position(reports[pid], names[pid]) for pid in checked]
positions = pd.DataFrame([p for p in positions if p], columns=["line", "word_before"])
positions["word_before"] = positions["word_before"].where(positions["word_before"].str.endswith((":", ".")), "(other word)")   # labels only, never name parts

print(positions["line"].value_counts(), "\n")
print(positions["word_before"].value_counts())

line
0     75
2     65
1      7
73     1
65     1
44     1
3      1
42     1
5      1
Name: count, dtype: int64 

word_before
Nome:           142
(other word)      8
Dr.               2
Téc.              1
Name: count, dtype: int64


For the 153 reports with the name, this shows on which line it appears and which word comes right before it (only labels are shown, any other word, which could be part of a name, is grouped as "other word"):

- **`line`:** almost always on line 0, 1 or 2, that is, in the header.

- **`word_before`:** in 142 reports the name comes right after the label `Nome:`. Most of the "other word" cases are also in the `Nome:` field, the first part of the name is shorter than 4 letters (e.g. a 3-letter first name), so the match starts at a later part of the name.

- **`Dr.` and `Téc.`:** the doctor or technician who signed the report shares a surname with the patient.

So the name can be removed by its position, the words after `Nome:`.

In [11]:
NAME_FIELD = re.compile(r"Nome:\s*([A-ZÀ-Ý][A-ZÀ-Ý' -]*[A-ZÀ-Ý])")   # 'Nome:' followed by words in capitals, on the same line

field = reports.str.extract(NAME_FIELD.pattern + r"\s*(\S*)")      # [0] = name (not printed), [1] = word after it
print(f"Reports with a 'Nome:' field: {field[0].notna().sum()} of {len(reports)}\n")
print("Words in the captured name:", field[0].str.split().str.len().value_counts().sort_index().to_dict(), "\n")
print(field[1].value_counts().head(10))

Reports with a 'Nome:' field: 175 of 256

Words in the captured name: {1.0: 1, 3.0: 8, 4.0: 35, 5.0: 57, 6.0: 47, 7.0: 20, 8.0: 6, 9.0: 1} 

1
:           168
º             2
NSC:          1
-01-005       1
ualidade      1
ltura:        1
ata           1
Name: count, dtype: int64


**A first attempt that does not work.** The pattern takes every word in capitals after `Nome:`:

- **175 reports** have a `Nome:` field.

- **Words captured:** mostly 4 to 7, more than a typical name.

- **Word after the capture:** in 168 reports it is just `:`, which means the pattern also swallowed the next label of the header (written in capitals) and stopped right before its colon. 

The other cases are the same mistake, `ata` is what is left of `Data`, `ltura:` of `Altura:`, `ualidade` of `Qualidade` and `º` of `Nº`.

Removing this would delete part of the header too, so the name has to stop at the next label.

**The working version.** `name_field` takes the words after `Nome:` and stops at the first word that is a label: a word ending in `:`, a word not in capitals, or `Nº`. In one report the name is glued to the next label (`...NSC:`, with no space), which is handled separately.

`deidentify_report` then:
1. replaces every occurrence of the name in the `Nome:` field with `[NAME]` (it can appear again later in the report);
2. replaces the numbers after `NSC:` (health number) and `Nº` (process number), and any other number with 6 or more digits (process or episode numbers without a label), with `[ID]`. Echocardiography measurements never have 6 or more digits;
3. for patients in the list, replaces any part of their name left anywhere in the text with `[NAME]`.

Both functions are defined in `src/multiscope.py`, so any notebook can use them.

In [12]:
clean_reports = pd.Series({pid: deidentify_report(r, names.get(pid)) for pid, r in reports.items()})

still_named = sum(has_name(clean_reports[pid], names[pid]) for pid in checked)
long_numbers = clean_reports.str.count(r"\b\d{6,}\b").sum()
caps_before_id = clean_reports.str.contains(r"(?<!\[)\b[A-ZÀ-Ý]{4,}\s+\[ID\]")   # a word in capitals right before an [ID]

print(f"Numbers with 6+ digits left: {long_numbers}")
print("Reports with a word in capitals right before an [ID] (review by hand):", sorted(clean_reports.index[caps_before_id]))
print(f"Reports still containing the patient's name: {still_named} of {len(checked)}")

Numbers with 6+ digits left: 0
Reports with a word in capitals right before an [ID] (review by hand): []
Reports still containing the patient's name: 0 of 225


- **Numbers with 6 or more digits (0):** a first version of `deidentify_report` left 20 of them, some right after a name that had not been replaced, the `Nome:` field was replaced only once, so a second copy of the name later in the report remained in reports of patients not in the list. The function now replaces the field everywhere and every number with 6 or more digits.

- **Words in capitals right before an `[ID]` (none):** that was the pattern where the leftover names appeared, and no report has it anymore.

- **Still containing the name (0 of 225):** in every report that can be checked against the list, the patient's name is gone.

In [13]:
unchecked = [pid for pid in reports.index if pid not in checked]
had_field = sum(name_field(reports[pid]) is not None for pid in unchecked)
uncertain = [pid for pid in unchecked if name_field(reports[pid]) is None]

print(f"Reports not in the list: {len(unchecked)} | with a 'Nome:' field (removed): {had_field}")
print("Reports to review by hand:", uncertain)

Reports not in the list: 31 | with a 'Nome:' field (removed): 27
Reports to review by hand: [292, 299, 300, 301]


The 31 reports of patients not in the list cannot be checked with the names, but 27 of them had a `Nome:` field, which was removed. The other 4 were reviewed by hand in the database:

- **Patient 292:** the cell is empty (only spaces), and it is marked in red in the database.

- **Patients 299, 300 and 301:** the cell only says "Sem relatório - Dr. [NAME] apenas realizou as medições", no report was written, only the measurements were taken. The name is the doctor's, not the patient's.

None of them is a real report, so they are treated as missing.

In [14]:
placeholder = clean_reports.str.strip().eq("") | clean_reports.str.contains("Sem relatório", case=False)
clean_reports = clean_reports[~placeholder]
del names                                       # the patient names are no longer needed

print(f"Placeholder reports removed: {placeholder.sum()} | real reports: {len(clean_reports)}")

Placeholder reports removed: 4 | real reports: 252


The reports were not de-identified, the patient's name appears in a `Nome:` field in the header of 175 of the 256 reports, usually followed by the health number (`NSC:`), and sometimes again later in the report, next to other identifier numbers.

`deidentify_report` replaces every occurrence of the name in the `Nome:` field with `[NAME]`, every identifier number (after `NSC:` or `Nº`, and any number with 6 or more digits) with `[ID]`, and, for patients in the patient list, any part of their name left anywhere in the text. After this, none of the 225 reports that can be checked against the list still contains the patient's name, and no identifier number is left in any report.

The other 31 reports belong to patients added after the list (14.01), so their names are unknown. 27 of them had their `Nome:` field removed. The remaining 4 are not real reports, patient 292 has an empty cell (marked in red in the database), and patients 299 to 301 only state that no report was written ("Sem relatório", only the measurements were taken). They are treated as missing, which leaves **252 de-identified reports**.

The de-identified reports are saved in `data/processed/`, so later notebooks can load them directly without reading the patient list again. The folder `data/` is not pushed to GitHub, and the file stays on this computer only.

In [15]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean_reports.rename_axis("ID").rename("report").to_pickle(REPORTS_FILE)
print(f"Saved {len(clean_reports)} de-identified reports to {REPORTS_FILE.name}")

Saved 252 de-identified reports to reports_deidentified.pkl


### 7. 12-lead ECGs
[[go back to the top]](#table-of-contents)

The supervisor said to keep the most recent 12-lead ECG of each patient, or to check which ones are useful. As a first step, this checks whether any patient has more than one file.

In [ ]:
ids_12 = pd.Series([int(p.name.split("_")[0]) for p in NEW_12_LEAD_DIR.glob("*.pkl")])
print(f"Files: {len(ids_12)} | patients: {ids_12.nunique()} | with more than one file: {sorted(ids_12[ids_12.duplicated()].unique())}")

Files: 133 | patients: 133 | with more than one file: []


Every patient has exactly one 12-lead ECG, so there is no choice to make between files. The next cell looks at their dates instead.

In [13]:
ecg12 = pd.DataFrame([{"patient": int(p.name.split("_")[0]),
                       "ecg_date": pd.to_datetime("_".join(p.stem.split("_")[-3:]), format="%d_%m_%Y")}
                      for p in NEW_12_LEAD_DIR.glob("*.pkl")])
evaluation = pd.to_datetime(db.set_index("ID")["EvaluationDate"], dayfirst=True, errors="coerce")
ecg12["days_from_evaluation"] = (ecg12["ecg_date"] - ecg12["patient"].map(evaluation)).dt.days

print(ecg12["ecg_date"].dt.year.value_counts().sort_index(), "\n")
ecg12["days_from_evaluation"].abs().describe().round(1)

ecg_date
2023    115
2025     18
Name: count, dtype: int64 



count       133.0
mean       5007.0
std       57735.9
min           0.0
25%           0.0
50%           0.0
75%           0.0
max      665843.0
Name: days_from_evaluation, dtype: float64

Each of the 133 patients with a 12-lead ECG has exactly one file, 115 from 2023 and 18 from 2025. The 2025 files are the newest extraction described in `README_12_lead_ECG.docx` (ECGs from 01/02/2025 to 11/06/2025, within 5 days of the evaluation). 

Most ECGs were taken on the evaluation day itself (median of 0 days), so the 2023 files are not outdated, they belong to the patients evaluated in 2023. All 133 are kept for now, what the supervisor meant by "the most recent ones" is still to be confirmed. A few evaluation dates give impossible differences (up to about 1 800 years), which points to typos in the `EvaluationDate` column (see the table below).

In [15]:
far = ecg12[ecg12["days_from_evaluation"].abs() > 5].assign(evaluation=lambda d: d["patient"].map(evaluation))
far.sort_values("patient")

,patient,ecg_date,days_from_evaluation,evaluation
74,3,2023-04-11,6,2023-04-05
61,24,2023-05-19,10,2023-05-09
99,69,2023-10-23,-8,2023-10-31
101,70,2023-10-23,-8,2023-10-31
41,199,2025-04-28,665843,202-04-23


Only five ECGs are more than 5 days from the evaluation. Patients 3, 24, 69 and 70 (6 to 10 days) are the ones listed in the README as outside the 5-day range. Patient 199 has an evaluation date of `202-04-23` in the database, a typo in the year (probably 2025), to be reported.